In [ ]:

## Notebook 1 — Exploration et prise en main de Spark
##💡 Créer un notebook 01_exploration.ipynb dans le dossier notebooks/. Toutes les questions de ce jalon se font dans ce notebook.


In [24]:
## Q1 — Créer la SparkSession - Créer une SparkSession avec le nom d'application 'TradeCorp ETL'. Afficher la version de Spark.
from pyspark.sql import SparkSession
import time
import pandas as pd

spark = SparkSession.builder \
    .appName("TradeCorp ETL") \
    .getOrCreate()

print(f"Version de Spark : {spark.version}")

Version de Spark : 3.5.0


In [25]:
## Q2 — Lire les 8 CSV - Lire les 8 fichiers CSV. Stocker chaque DataFrame dans une variable.
path = "/home/jovyan/data/raw/"

df_categories = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{path}categories.csv")
df_customers = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{path}customers.csv")
df_employees = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{path}employees.csv")
df_order_details = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{path}order_details.csv")
df_orders = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{path}orders.csv")
df_products = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{path}products.csv")
df_shippers = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{path}shippers.csv")
df_suppliers = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{path}suppliers.csv")

In [26]:
## Q3 — Explorer le schema Pour chaque DataFrame, afficher le schema des données. Identifier les types de colonnes inférés automatiquement.
dfs = {
    "categories": df_categories,
    "customers": df_customers,
    "employees": df_employees,
    "order_details": df_order_details,
    "orders": df_orders,
    "products": df_products,
    "shippers": df_shippers,
    "suppliers": df_suppliers
}

for nom, df in dfs.items():
    print(f"=== Schéma : {nom} ===")
    df.printSchema()

=== Schéma : categories ===
root
 |-- category_id: integer (nullable = true)
 |-- category_name: string (nullable = true)
 |-- description: string (nullable = true)
 |-- picture: string (nullable = true)

=== Schéma : customers ===
root
 |-- customer_id: string (nullable = true)
 |-- company_name: string (nullable = true)
 |-- contact_name: string (nullable = true)
 |-- contact_title: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- country: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- fax: string (nullable = true)

=== Schéma : employees ===
root
 |-- employee_id: integer (nullable = true)
 |-- last_name: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- title: string (nullable = true)
 |-- title_of_courtesy: string (nullable = true)
 |-- birth_date: date (nullable = true)
 |-- hire_date: date (nullable = t

In [27]:
## Q4 — Afficher les données - Pour chaque DataFrame, afficher les 5 premières lignes. Observer la structure des données.
for nom, df in dfs.items():
    print(f"=== Données : {nom} ===")
    df.show(5, truncate=False)

=== Données : categories ===
+-----------+--------------+----------------------------------------------------------+-------+
|category_id|category_name |description                                               |picture|
+-----------+--------------+----------------------------------------------------------+-------+
|1          |Beverages     |Soft drinks, coffees, teas, beers, and ales               |NULL   |
|2          |Condiments    |Sweet and savory sauces, relishes, spreads, and seasonings|NULL   |
|3          |Confections   |Desserts, candies, and sweet breads                       |NULL   |
|4          |Dairy Products|Cheeses                                                   |NULL   |
|5          |Grains/Cereals|Breads, crackers, pasta, and cereal                       |NULL   |
+-----------+--------------+----------------------------------------------------------+-------+
only showing top 5 rows

=== Données : customers ===
+-----------+----------------------------------+------

In [28]:
## Q5 — Compter les lignes - Compter le nombre de lignes de chaque DataFrame avec. Créer un tableau récapitulatif.
recap_lignes = [(nom, df.count()) for nom, df in dfs.items()]

# Affichage sous forme de tableau formaté
print(f"{'DataFrame':<15} | {'Nombre de lignes'}")
print("-" * 35)
for nom, count in recap_lignes:
    print(f"{nom:<15} | {count}")


DataFrame       | Nombre de lignes
-----------------------------------
categories      | 8
customers       | 91
employees       | 9
order_details   | 2155
orders          | 830
products        | 77
shippers        | 6
suppliers       | 29


In [29]:
## Q6 — Statistiques descriptives - Sur df_orders et df_products essayer d’obtenir les statistiques suivantes : min, max, mean, stddev.

# Statistiques descriptives pour df_orders
print("=== Statistiques : df_orders ===")
df_orders.describe().show()

# Statistiques descriptives pour df_products
print("=== Statistiques : df_products ===")
df_products.describe().show()

=== Statistiques : df_orders ===
+-------+-----------------+-----------+------------------+------------------+------------------+--------------------+--------------------+---------+-----------+------------------+------------+
|summary|         order_id|customer_id|       employee_id|          ship_via|           freight|           ship_name|        ship_address|ship_city|ship_region|  ship_postal_code|ship_country|
+-------+-----------------+-----------+------------------+------------------+------------------+--------------------+--------------------+---------+-----------+------------------+------------+
|  count|              830|        830|               830|               830|               830|                 830|                 830|      830|        323|               811|         830|
|   mean|          10662.5|       NULL| 4.403614457831325|2.0072289156626506| 78.24420481927719|                NULL|                NULL|     NULL|       NULL|39975.067357512955|        NULL|
| 

In [11]:
## Q7 — Lazy evaluation - Expliquer dans une cellule Markdown ce qu'est la lazy evaluation dans Spark. Quelle est la différence entre une transformation et une action ? Donner 3 exemples de chaque.
"""
La lazy evaluation (ou évaluation paresseuse) dans Spark signifie que les opérations sur les DataFrames (ou RDDs) ne sont pas exécutées immédiatement au moment où tu les écris. Spark se contente de construire un graphe d'exécution (un graphe de dépendance logique ou DAG - Directed Acyclic Graph) en enregistrant les étapes successives. Rien ne se calcule physiquement tant qu'une action explicite n'est pas déclenchée. Cela permet à Spark d'optimiser l'ensemble du plan de traitement avant de lancer les calculs sur le cluster.

Transformations vs Actions
Transformations : Opérations paresseuses qui prennent un DataFrame en entrée et renvoient un nouveau DataFrame transformé. Elles ne déclenchent aucun calcul immédiat.

Actions : Opérations qui déclenchent l'exécution effective du DAG pour produire un résultat renvoyé au driver ou écrit dans un stockage externe.

Exemples
3 exemples de transformations :

select() : Sélectionne des colonnes spécifiques d'un DataFrame.

filter() / where() : Filtre les lignes selon une condition logique.

join() : Joint deux DataFrames sur une clé commune.

3 exemples d'actions :

show() : Affiche un aperçu des premières lignes dans la console.

count() : Compte et renvoie le nombre total de lignes.

write (ex: write.parquet()) : Sauvegarde les données sur un système de fichiers ou un stockage distant.
"""

"\nLa lazy evaluation (ou évaluation paresseuse) dans Spark signifie que les opérations sur les DataFrames (ou RDDs) ne sont pas exécutées immédiatement au moment où tu les écris. Spark se contente de construire un graphe d'exécution (un graphe de dépendance logique ou DAG - Directed Acyclic Graph) en enregistrant les étapes successives. Rien ne se calcule physiquement tant qu'une action explicite n'est pas déclenchée. Cela permet à Spark d'optimiser l'ensemble du plan de traitement avant de lancer les calculs sur le cluster.\n\nTransformations vs Actions\nTransformations : Opérations paresseuses qui prennent un DataFrame en entrée et renvoient un nouveau DataFrame transformé. Elles ne déclenchent aucun calcul immédiat.\n\nActions : Opérations qui déclenchent l'exécution effective du DAG pour produire un résultat renvoyé au driver ou écrit dans un stockage externe.\n\nExemples\n3 exemples de transformations :\n\nselect() : Sélectionne des colonnes spécifiques d'un DataFrame.\n\nfilter(

In [30]:
## Q8 — Spark UI - Ouvrir http://localhost:4040 dans le navigateur. Observer les jobs exécutés. Identifier ce que représentent les stages et tasks.
print(spark.sparkContext.uiWebUrl)
print(spark.sparkContext.applicationId)
"""
Jobs : Chaque ligne de l'interface correspond à un Job déclenché par une action PySpark (comme count(), show() ou la lecture de fichiers csv).
La liste montre l'ensemble des requêtes exécutées séquentiellement par ton notebook.

Stages : Un job est découpé en un ou plusieurs Stages (indiqués par la colonne Stages: Succeeded/Total). Un nouveau stage est créé lorsqu'une transformation large (wide transformation) nécessite un échange de données (shuffle)
entre les partitions.

Tasks : Chaque stage est exécuté sous forme de Tasks (tâches unitaires dans la colonne Tasks). Une tâche correspond au travail élémentaire effectué en parallèle sur une partition de données par un cœur de calcul.
La valeur 1/1 montre que le traitement s'est fait sur une seule partition de travail.

"""

http://345be98db1ac:4040
local-1787921187276


"\nJobs : Chaque ligne de l'interface correspond à un Job déclenché par une action PySpark (comme count(), show() ou la lecture de fichiers csv).\nLa liste montre l'ensemble des requêtes exécutées séquentiellement par ton notebook.\n\nStages : Un job est découpé en un ou plusieurs Stages (indiqués par la colonne Stages: Succeeded/Total). Un nouveau stage est créé lorsqu'une transformation large (wide transformation) nécessite un échange de données (shuffle)\nentre les partitions.\n\nTasks : Chaque stage est exécuté sous forme de Tasks (tâches unitaires dans la colonne Tasks). Une tâche correspond au travail élémentaire effectué en parallèle sur une partition de données par un cœur de calcul.\nLa valeur 1/1 montre que le traitement s'est fait sur une seule partition de travail.\n\n"

In [31]:
## Q9 — Spark vs Pandas - Lire le fichier orders.csv avec Pandas et mesurer le temps. Puis avec Spark. Comparer. Expliquer quand Spark est plus pertinent que Pandas.

# 1. Mesure du temps avec Pandas
start_time = time.time()
pdf_orders = pd.read_csv("/home/jovyan/data/raw/orders.csv")
pandas_duration = time.time() - start_time
print(f"Temps de lecture avec Pandas : {pandas_duration:.4f} secondes")

# 2. Mesure du temps avec Spark (en forçant une action comme .count() pour déclencher le chargement)
start_time = time.time()
df_orders_spark = spark.read.option("header", "true").option("inferSchema", "true").csv("/home/jovyan/data/raw/orders.csv")
spark_count = df_orders_spark.count()  # Action pour exécuter réellement le chargement
spark_duration = time.time() - start_time
print(f"Temps de lecture avec Spark : {spark_duration:.4f} secondes")


"""
Comparaison des performances : Pandas s'avère beaucoup plus rapide sur les petits volumes de données car il charge le fichier directement en mémoire locale sans surcoût de planification.
Spark présente un temps d'exécution légèrement supérieur sur ce même fichier en raison de la complexité de son moteur d'exécution distribuée et de la gestion des tâches.

Pertinence de Spark : Spark devient indispensable et plus pertinent que Pandas lorsque le volume de données dépasse la capacité de la mémoire RAM d'une seule machine (Big Data), ou lorsque
les traitements doivent être distribués horizontalement sur un cluster de plusieurs nœuds pour paralléliser la charge de calcul.

"""

Temps de lecture avec Pandas : 0.0053 secondes
Temps de lecture avec Spark : 0.1634 secondes


"\nComparaison des performances : Pandas s'avère beaucoup plus rapide sur les petits volumes de données car il charge le fichier directement en mémoire locale sans surcoût de planification.\nSpark présente un temps d'exécution légèrement supérieur sur ce même fichier en raison de la complexité de son moteur d'exécution distribuée et de la gestion des tâches.\n\nPertinence de Spark : Spark devient indispensable et plus pertinent que Pandas lorsque le volume de données dépasse la capacité de la mémoire RAM d'une seule machine (Big Data), ou lorsque\nles traitements doivent être distribués horizontalement sur un cluster de plusieurs nœuds pour paralléliser la charge de calcul.\n\n"

In [32]:
## Q10 — Colonnes et types - Afficher la liste des colonnes de df_orders avec df.columns. Afficher les types avec df.dtypes. Identifier les colonnes qui nécessitent un cast de type.
# Affichage de la liste des colonnes
print("--- Colonnes ---")
print(df_orders_spark.columns)

# Affichage des types de chaque colonne
print("\n--- Types de données ---")
display(df_orders_spark.dtypes)
# (df_orders_spark.dtypes)

"""
Colonnes correctement typées : order_id et employee_id sont en entiers (int), freight est en nombre à virgule (double), et les dates (order_date, required_date, shipped_date) sont bien reconnues au format date grâce à l'inférence de schéma.

Colonnes nécessitant potentiellement un cast :

ship_postal_code : Actuellement en string, elle pourrait être castée en entier (int ou long) si l'on souhaite effectuer des traitements numériques stricts, 
bien qu'en pratique elle reste souvent en texte pour préserver les zéros initiaux (codes postaux internationaux).

Les autres identifiants (customer_id) ou attributs textuels restent logiquement en string.
"""

--- Colonnes ---
['order_id', 'customer_id', 'employee_id', 'order_date', 'required_date', 'shipped_date', 'ship_via', 'freight', 'ship_name', 'ship_address', 'ship_city', 'ship_region', 'ship_postal_code', 'ship_country']

--- Types de données ---


[('order_id', 'int'),
 ('customer_id', 'string'),
 ('employee_id', 'int'),
 ('order_date', 'date'),
 ('required_date', 'date'),
 ('shipped_date', 'date'),
 ('ship_via', 'int'),
 ('freight', 'double'),
 ('ship_name', 'string'),
 ('ship_address', 'string'),
 ('ship_city', 'string'),
 ('ship_region', 'string'),
 ('ship_postal_code', 'string'),
 ('ship_country', 'string')]

"\nColonnes correctement typées : order_id et employee_id sont en entiers (int), freight est en nombre à virgule (double), et les dates (order_date, required_date, shipped_date) sont bien reconnues au format date grâce à l'inférence de schéma.\n\nColonnes nécessitant potentiellement un cast :\n\nship_postal_code : Actuellement en string, elle pourrait être castée en entier (int ou long) si l'on souhaite effectuer des traitements numériques stricts, \nbien qu'en pratique elle reste souvent en texte pour préserver les zéros initiaux (codes postaux internationaux).\n\nLes autres identifiants (customer_id) ou attributs textuels restent logiquement en string.\n"